<a href="https://colab.research.google.com/github/Mukunzijames/Formative3/blob/main/Part_2_Bayesian_Probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2 - Bayesian Probability

**1. Keyword Selection**

As a group, choose 2–4 keywords that you believe indicate positive sentiment and 2–4 keywords that indicate negative sentiment.

Positive Keywords: `great, amazing, excellent`

Negative Keywords: `bad, waste, horrible`

**2. Choice of Conditional Probability**

**`Negative|Keyword`**

**3. Computation and Presentation**

Probabilities:

* Prior: P(Positive)

* Likelihood: P(keyword|Positive)

* Marginal:  P(keyword)

* Posterior:  P(Positive|keyword)

Loading the dataset

**4. Implementation**

Implementing Bayes’ Theorem in Python to compute the posterior probability for each keyword based on your selected dataset.

In [15]:
# importing required libraries
%pip install -q pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns


In [16]:
# Loading the dataset & checking for blanck files
df = pd.read_csv("IMDB Dataset.csv", engine='python', on_bad_lines='skip')
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSample rows:")
print(df.head(30))
print(df.isnull().sum())


Dataset shape: (50000, 2)
Columns: ['review', 'sentiment']

Sample rows:
                                               review sentiment
0   One of the other reviewers has mentioned that ...  positive
1   A wonderful little production. <br /><br />The...  positive
2   I thought this was a wonderful way to spend ti...  positive
3   Basically there's a family where a little boy ...  negative
4   Petter Mattei's "Love in the Time of Money" is...  positive
5   Probably my all-time favorite movie, a story o...  positive
6   I sure would like to see a resurrection of a u...  positive
7   This show was an amazing, fresh & innovative i...  negative
8   Encouraged by the positive comments about this...  negative
9   If you like original gut wrenching laughter yo...  positive
10  Phil the Alien is one of those quirky films wh...  negative
11  I saw this movie when I was about 12 when it c...  negative
12  So im not a big fan of Boll's work but then ag...  negative
13  The cast played Shakespear

In [17]:
# Check for missing data in each column
print("\nChecking for missing values:")
for col in df.columns:
    missing_count = df[col].isna().sum()
    print(f"  {col}: {missing_count} missing")

# Convert all reviews to lowercase for easier keyword matching
df['review_clean'] = df['review'].apply(lambda x: x.lower())

# Make sure sentiment labels are consistent
df['sentiment_clean'] = df['sentiment'].str.lower().str.strip()

# Count how many reviews are positive or negative
positive_count = df[df['sentiment_clean'] == 'positive'].shape[0]
negative_count = df[df['sentiment_clean'] == 'negative'].shape[0]
total_count = df.shape[0]

# Print sentiment distribution
print("\nSentiment Distribution:")
print(f"  Positive: {positive_count} ({positive_count/total_count*100:.0f}%)")
print(f"  Negative: {negative_count} ({negative_count/total_count*100:.0f}%)")

# Calculate base probabilities (priors)
P_positive = positive_count / total_count
P_negative = negative_count / total_count

print("\nBase Probabilities (Priors):")
print(f"  P(Positive) = {P_positive:.4f} ({P_positive*100:.0f}%)")
print(f"  P(Negative) = {P_negative:.4f} ({P_negative*100:.0f}%)")


Checking for missing values:
  review: 0 missing
  sentiment: 0 missing

Sentiment Distribution:
  Positive: 25000 (50.00%)
  Negative: 25000 (50.00%)

Base Probabilities (Priors):
  P(Positive) = 0.5000 (50.00%)
  P(Negative) = 0.5000 (50.00%)


In [18]:

# Analysis Configuration
positive_keywords = ['great', 'amazing', 'excellent']
negative_keywords = ['bad', 'waste', 'horrible']

# Target sentiment for conditional probability
target_sentiment = 'negative'

print("\nAnalysis Configuration:")
print(f"  Target sentiment: {target_sentiment.capitalize()}")
# print(f"  Positive keywords: {', '.join(positive_keywords)}")
print(f"  Negative keywords: {', '.join(negative_keywords)}")


Analysis Configuration:
  Target sentiment: Negative
  Positive keywords: great, amazing, excellent
  Negative keywords: bad, waste, horrible


In [19]:
# Keyword Detection (Custom)
def contains_keyword(text, keyword):
    """Check if a keyword exists as a whole word in the text (case-insensitive)."""
    pattern = r'\b' + re.escape(keyword.lower()) + r'\b'
    return bool(re.search(pattern, str(text).lower()))

all_keywords = positive_keywords + negative_keywords

print(f"\nSearching for {len(all_keywords)} keywords in {len(df):,} reviews...")

for keyword in all_keywords:
    column_name = f"has_{keyword}"

    # Apply keyword detection on cleaned review column
    df[column_name] = df['review_clean'].apply(lambda x: contains_keyword(x, keyword))

    # Count occurrences of the keyword
    count = df[column_name].sum()
    percentage = (count / len(df)) * 100

    # Determine keyword type
    keyword_type = "POSITIVE" if keyword in positive_keywords else "NEGATIVE"

    print(f"  [{keyword_type}] '{keyword}': {count:,} occurrences ({percentage:.2f}%)")

    # Count how many of these keyword reviews match the target sentiment
    sentiment_count = df[(df[column_name]) & (df['sentiment_clean'] == target_sentiment)].shape[0]
    print(f"      Reviews with target sentiment '{target_sentiment}': {sentiment_count:,}")


Searching for 6 keywords in 50,000 reviews...
  [POSITIVE] 'great': 12,576 occurrences (25.15%)
      Reviews with target sentiment 'negative': 4,098
  [POSITIVE] 'amazing': 2,158 occurrences (4.32%)
      Reviews with target sentiment 'negative': 479
  [POSITIVE] 'excellent': 3,552 occurrences (7.10%)
      Reviews with target sentiment 'negative': 684
  [NEGATIVE] 'bad': 11,788 occurrences (23.58%)
      Reviews with target sentiment 'negative': 8,827
  [NEGATIVE] 'waste': 2,534 occurrences (5.07%)
      Reviews with target sentiment 'negative': 2,359
  [NEGATIVE] 'horrible': 2,101 occurrences (4.20%)
      Reviews with target sentiment 'negative': 1,775


In [20]:
# Tokenize reviews into words
df["tokens"] = df["review_clean"].apply(
    lambda x: re.findall(r"\b\w+\b", x)
)

# All words (entire dataset)
all_words = [word for tokens in df["tokens"] for word in tokens]
all_word_counts = Counter(all_words)
total_words = sum(all_word_counts.values())

# Words in negative reviews
negative_reviews = df[df["sentiment_clean"] == "negative"]
negative_words = [word for tokens in negative_reviews["tokens"] for word in tokens]
negative_word_counts = Counter(negative_words)
total_words_in_negative = sum(negative_word_counts.values())

#

### Bayesian Computation Function

In [21]:
def compute_bayes(keyword):
    """
    Computes:
    P(keyword)
    P(keyword | Negative)
    P(Negative | keyword)
    """

    count_keyword = all_word_counts[keyword]
    count_keyword_in_negative = negative_word_counts[keyword]

    # Avoid division by zero
    if count_keyword == 0:
        return 0, 0, 0

    p_keyword = count_keyword / total_words
    p_keyword_given_negative = count_keyword_in_negative / total_words_in_negative

    # Bayes Theorem
    p_negative_given_keyword = (
        p_keyword_given_negative * P_negative
    ) / p_keyword

    return p_keyword, p_keyword_given_negative, p_negative_given_keyword

`For Selected Keywords`

In [22]:
keywords = ["waste", "horrible", "bad"]

results = []

for keyword in keywords:
    p_keyword, p_keyword_given_negative, p_negative_given_keyword = compute_bayes(keyword)

    results.append({
        "Keyword": keyword,
        "P(Negative)": P_negative,
        "P(Keyword)": p_keyword,
        "P(Keyword|Negative)": p_keyword_given_negative,
        "P(Negative|Keyword)": p_negative_given_keyword
    })


Results Summary

In [23]:

probability_df = pd.DataFrame(results)

print("\nBayesian Probability Results:")
print(probability_df)


Bayesian Probability Results:
    Keyword  P(Negative)  P(Keyword)  P(Keyword|Negative)  P(Negative|Keyword)
0     waste          0.5    0.000233             0.000438             0.941476
1  horrible          0.5    0.000211             0.000367             0.869792
2       bad          0.5    0.001542             0.002473             0.801675
